# Jobsite Twin — Synthetic Construction Data

Populates a Fabric Lakehouse with a synthetic construction portfolio of 20 projects, 116 zones, ~347 tasks, and ~1041 cost rows — all deterministic (SEED=42) so anyone running this notebook gets identical output.

**Safe to run:**

- ✅ **First-time users:** creates 4 fresh Delta tables (`dbo.projects`, `dbo.tasks`, `dbo.costs`, `dbo.locations`) in your attached Lakehouse.
- ✅ **Re-runs:** safely replaces those tables with the same data (no accumulation, no duplicates).

**To run:** attach any Lakehouse to this notebook and click **Run all**. The 3 validation cells at the end (Section 9) will assert the output is correct — if any assertion fails, execution stops with a clear error.


# 🏗️ Synthetic Construction Project Intelligence Dataset

**A reusable synthetic data generator for construction project intelligence — built for Microsoft Fabric (Lakehouse).**

---

## Overview

This notebook generates a **fully synthetic, realistic construction project dataset** using PySpark and writes it to the Fabric Lakehouse as Delta tables. 

### What this dataset enables

| Capability | Description |
| --- | --- |
| 📈 **Project analytics** | Portfolio-level KPIs, status rollups, and risk scoring |
| 🗓️ **Scheduling analysis** | P6-style task dependency chains with cascading delays |
| 💲 **Cost analysis** | ERP-style planned vs. actual cost with overrun modeling |
| 🧭 **Spatial mapping** | 3D zone coordinates for app / digital-twin visualization |
| ⚠️ **Risk modeling** | Composite risk scores that reflect schedule **and** cost pressure |

### Designed for downstream use in

- **Power BI** — semantic models and dashboards
- **Fabric Apps** — operational and 3D visualization scenarios
- **AI / Copilot** — grounding data for natural-language analytics

### How it is structured

The notebook is **modular by design**. Each table is produced by its own reusable function (`create_projects`, `create_tasks`, `create_costs`, `create_locations`), realism is layered on top, and a final **refactor section** consolidates everything into a single clean entry point.

> **Prerequisite:** Attach a **Lakehouse** to this notebook before running. The Delta tables are written to the default Lakehouse via `saveAsTable()`.

## 1. Setup & Imports

We rely exclusively on **PySpark DataFrames** for data generation (no Pandas). A single fixed random seed (`SEED`) guarantees that every run produces the **same** dataset — essential for reproducible demos, tests, and documentation.

Why a seed matters: reproducibility means screenshots, dashboards, and validation numbers stay stable across runs and across machines.

In [ ]:
# Core PySpark imports — DataFrame API only (no Pandas for generation)
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F

# In Fabric, `spark` is provided automatically. This fallback keeps the notebook
# runnable in any Spark environment.
spark = SparkSession.builder.getOrCreate()

# Fixed seed for fully reproducible synthetic data.
SEED = 42

# Generation volume controls (kept as constants so the dataset is easy to scale).
NUM_PROJECTS = 20      # ~20 projects
MIN_TASKS = 10         # 10-25 tasks per project
MAX_TASKS = 25
MIN_ZONES = 4          # spatial zones per project
MAX_ZONES = 8

print("\u2705 Spark session ready and configuration loaded")


## 1.1 Task–Zone Placement Logic

Earlier revisions placed each task in a zone by a **round-robin index**
(`zone_idx = (task_seq - 1) % zone_count`). Because a task's zone was just its
position in the sequence modulo the project's zone count, the result was
physically nonsensical — **Excavation on Floor 2, Roofing in the Basement,
Landscaping indoors**.

To make the 3D digital twin read like a real building, we map every **task type**
to the **zones where that work realistically happens** via `TASK_ZONE_AFFINITY`,
and each task is placed in one of its affinity zones **that actually exists in the
project**. The pick is deterministic (seeded `xxhash64`), so the dataset stays
fully reproducible while still spreading work across floors for visual variety.

`ZONE_HEIGHT` powers a **nearest-by-height fallback**: when a project contains
none of a task's affinity zones (e.g. Roofing on a project with no Roof zone),
the task falls to the reachable zone closest in height to its target — so
**Roofing → Floor 3** and **Landscaping → Basement** rather than a random floor.

> Only the **7 zones** and **24 tasks** that actually materialize are listed —
> zone counts top out at 7 and task counts at 24 with the current volume settings.


In [ ]:
# Reachable spatial zones (bottom -> top). Only these 7 materialize: zone_count
# tops out at 7, so no 8th zone is ever generated.
ZONE_NAMES = [
    "Floor 1", "Floor 2", "Floor 3", "Basement",
    "Mechanical Room", "Exterior", "Roof",
]

# Reachable task types (P6-style order). Only these 24 materialize: task_count
# tops out at 24, so no 25th task is ever generated.
TASK_NAMES = [
    "Site Mobilization", "Excavation", "Foundation", "Structural Steel", "Concrete Pour",
    "Framing", "Roofing", "MEP Rough-In", "Electrical", "Plumbing", "HVAC Installation",
    "Drywall", "Interior Finishes", "Exterior Cladding", "Glazing", "Flooring", "Painting",
    "Landscaping", "Fire Protection", "Elevator Installation", "Inspections",
    "Commissioning", "Punch List", "Final Handover",
]

# Physical height rank per zone, used only by the nearest-by-height fallback below.
# Basement/Exterior are at grade (0); floors climb; Roof is the top.
ZONE_HEIGHT = {
    "Basement": 0,
    "Exterior": 0,
    "Floor 1": 1,
    "Floor 2": 2,
    "Floor 3": 3,
    "Mechanical Room": 3,
    "Roof": 4,
}

# Construction placement logic: each task type -> the zones where that work
# realistically happens. A task is placed in one of its affinity zones that
# actually exists in the project (see create_tasks). Grouped by build phase.
TASK_ZONE_AFFINITY = {
    # Substructure — below-grade work
    "Site Mobilization":     ["Basement"],
    "Excavation":            ["Basement"],
    "Foundation":            ["Basement"],
    # Superstructure — structural frame rising through the building
    "Structural Steel":      ["Basement", "Floor 1", "Floor 2", "Floor 3"],
    "Concrete Pour":         ["Basement", "Floor 1", "Floor 2", "Floor 3"],
    "Framing":               ["Floor 1", "Floor 2", "Floor 3"],
    "Roofing":               ["Roof"],
    # MEP / building systems — plant room, risers and every occupied floor
    "MEP Rough-In":          ["Basement", "Floor 1", "Floor 2", "Floor 3", "Mechanical Room"],
    "Electrical":            ["Basement", "Floor 1", "Floor 2", "Floor 3", "Mechanical Room"],
    "Plumbing":              ["Basement", "Floor 1", "Floor 2", "Floor 3", "Mechanical Room"],
    "HVAC Installation":     ["Basement", "Floor 1", "Floor 2", "Floor 3", "Mechanical Room"],
    "Fire Protection":       ["Basement", "Floor 1", "Floor 2", "Floor 3", "Mechanical Room"],
    # Envelope — facade work seen from outside and the perimeter of each floor
    "Exterior Cladding":     ["Exterior", "Floor 1", "Floor 2", "Floor 3"],
    "Glazing":               ["Exterior", "Floor 1", "Floor 2", "Floor 3"],
    # Interior finishes — occupied floors only
    "Drywall":               ["Floor 1", "Floor 2", "Floor 3"],
    "Interior Finishes":     ["Floor 1", "Floor 2", "Floor 3"],
    "Flooring":              ["Floor 1", "Floor 2", "Floor 3"],
    "Painting":              ["Floor 1", "Floor 2", "Floor 3"],
    # Sitework — grounds only
    "Landscaping":           ["Exterior"],
    # Vertical transport — spans the full height of the building
    "Elevator Installation": ["Basement", "Floor 1", "Floor 2", "Floor 3", "Mechanical Room", "Roof"],
    # Closeout / QA — inspectable occupied spaces; handover touches everything
    "Inspections":           ["Floor 1", "Floor 2", "Floor 3"],
    "Commissioning":         ["Floor 1", "Floor 2", "Floor 3"],
    "Punch List":            ["Basement", "Floor 1", "Floor 2", "Floor 3", "Mechanical Room", "Exterior", "Roof"],
    "Final Handover":        ["Basement", "Floor 1", "Floor 2", "Floor 3", "Mechanical Room", "Exterior", "Roof"],
}

print(f"\u2705 Placement config loaded: {len(ZONE_NAMES)} zones, {len(TASK_NAMES)} tasks, "
      f"{len(TASK_ZONE_AFFINITY)} affinity rules")


In [ ]:
# Small helper: build a Spark array<string> literal from a Python list.
# Used to pick realistic values by index across the dataset.
def lit_array(values):
    return F.array(*[F.lit(v) for v in values])


# Pick an element from a literal array using a uniform random column (seeded).
# element_at is 1-based, so we add 1 to the zero-based random index.
def pick_random(values, seed_offset):
    idx = (F.rand(SEED + seed_offset) * len(values)).cast("int") + 1
    return F.element_at(lit_array(values), idx)


print("\u2705 Reusable helper functions defined")

## 2. Data Model Design

The dataset follows a simple **star-style** model centered on the **project**. This keeps it intuitive for Power BI modeling and easy to reason about for AI scenarios.

```
                 ┌───────────────┐
                 │   projects    │   one row per construction project
                 └───────┬───────┘
                         │ project_id
        ┌────────────────┼────────────────┐
        │                │                │
  ┌─────▼─────┐    ┌──────▼──────┐   ┌─────▼──────┐
  │   tasks   │    │  locations  │   │   costs    │
  └─────┬─────┘    └─────────────┘   └─────┬──────┘
        │ task_id ───────────────────────── │
        │ location_id ──► locations.location_id
```

### Tables & relationships

| Table | Grain | Key relationships |
| --- | --- | --- |
| **projects** | one row per project | parent of all other tables via `project_id` |
| **tasks** | one row per scheduled task | `project_id` → projects; `dependency_task_id` → tasks (self-join); `location_id` → locations |
| **costs** | one row per task × cost category | `project_id` → projects; `task_id` → tasks |
| **locations** | one row per project zone | `project_id` → projects |

### Why this model

- **Projects** anchor every analytical question ("which projects are at risk?").
- **Tasks** carry the **P6-style schedule** — each task depends on the one before it, so delays **cascade**.
- **Costs** mirror an **ERP** breakdown (labor / materials / equipment) and react to schedule slippage.
- **Locations** provide **x / y / z** coordinates so tasks can be placed in a **3D** scene.

## 3. Synthetic Data Generation

Each table is produced by a dedicated, reusable function. This avoids duplication, keeps the logic testable, and makes it trivial to regenerate or scale the dataset.

We deliberately carry a few **helper columns** (`delay_factor`, `is_outlier`, `task_delay_days`) between functions so that **schedule and cost stay consistent** — a delayed project produces delayed tasks, which in turn produce cost overruns. These helper columns are dropped before the tables are persisted.

### 3.1 `create_projects()`

Generates ~20 projects with realistic, **fully generic** names (no real organizations), US city locations, planned vs. actual end dates, a status, and a composite **risk score** that blends schedule pressure and cost uncertainty. The first two projects are flagged as **outliers** to inject extreme delays later.

In [ ]:
def create_projects(spark: SparkSession, num_projects: int = NUM_PROJECTS) -> DataFrame:
    """Generate the projects table. Carries helper columns (delay_factor, is_outlier)
    used by downstream tasks/costs generation; these are dropped before saving."""

    project_types = ["commercial", "residential", "healthcare", "infrastructure"]
    cities = [
        "Austin, TX", "Denver, CO", "Seattle, WA", "Phoenix, AZ", "Columbus, OH",
        "Raleigh, NC", "Nashville, TN", "Portland, OR", "Tampa, FL", "Kansas City, MO",
        "Salt Lake City, UT", "Charlotte, NC", "Boise, ID", "Madison, WI", "Omaha, NE",
    ]
    name_a = [
        "Summit", "Riverside", "Gateway", "Cornerstone", "Horizon", "Pinnacle",
        "Heritage", "Lakeview", "Northgate", "Meadowbrook", "Ironwood", "Brookfield",
    ]
    name_b = [
        "Medical Center", "Office Tower", "Residences", "Logistics Hub", "Community Center",
        "Transit Station", "Business Park", "Apartments", "Distribution Center", "Civic Plaza",
    ]

    df = spark.range(0, num_projects).withColumnRenamed("id", "idx")

    # Identity & descriptive attributes
    df = df.withColumn(
        "project_id", F.concat(F.lit("PRJ-"), F.lpad((F.col("idx") + 1).cast("string"), 4, "0"))
    )
    df = df.withColumn("project_name", F.concat(pick_random(name_a, 2), F.lit(" "), pick_random(name_b, 3)))
    df = df.withColumn("project_type", pick_random(project_types, 0))
    df = df.withColumn("location", pick_random(cities, 1))

    # Outlier flag (first two projects) -> extreme delays downstream
    df = df.withColumn("is_outlier", F.col("idx") < 2)

    # delay_factor in ~[0,1]; boosted >1 for outliers to create extreme slippage
    rnd_delay = F.rand(SEED + 6)
    df = df.withColumn(
        "delay_factor",
        F.when(F.col("is_outlier"), F.lit(1.0) + rnd_delay).otherwise(rnd_delay),
    )

    # Schedule: start within 2023, planned duration 180-540 days, delay scaled by delay_factor
    rnd_start = F.rand(SEED + 4)
    rnd_dur = F.rand(SEED + 5)
    df = df.withColumn("start_date", F.date_add(F.lit("2023-01-01").cast("date"), (rnd_start * 300).cast("int")))
    df = df.withColumn("planned_duration", (F.lit(180) + (rnd_dur * 360)).cast("int"))
    df = df.withColumn("planned_end_date", F.date_add(F.col("start_date"), F.col("planned_duration")))
    df = df.withColumn("delay_days", (F.col("delay_factor") * 120).cast("int"))
    df = df.withColumn("actual_end_date", F.date_add(F.col("planned_end_date"), F.col("delay_days")))

    # Status derived from schedule pressure
    df = df.withColumn(
        "status",
        F.when(F.col("delay_factor") < 0.33, F.lit("On Track"))
         .when(F.col("delay_factor") < 0.66, F.lit("At Risk"))
         .otherwise(F.lit("Delayed")),
    )

    # Risk score (0-100): blends schedule slippage and cost/duration uncertainty
    df = df.withColumn(
        "risk_score",
        F.round(F.least(F.lit(100.0), F.col("delay_factor") * 60 + rnd_dur * 40), 1),
    )

    return df.select(
        "project_id", "project_name", "project_type", "location", "start_date",
        "planned_end_date", "actual_end_date", "status", "risk_score",
        "delay_factor", "is_outlier",  # helper columns (dropped before save)
    )


print("\u2705 create_projects() defined")

### 3.2 `create_locations()`

Each project gets **4–8 spatial zones** (Floor 1, Mechanical Room, Exterior, …). The `z_coord` is derived from the zone sequence so floors stack vertically — making the data immediately usable for **3D / digital-twin** visualizations.

In [ ]:
def create_locations(spark: SparkSession, projects_df: DataFrame) -> DataFrame:
    """Generate spatial zones per project with 3D coordinates."""

    zones = ZONE_NAMES  # 7 reachable zones defined near the top of the notebook

    base = projects_df.select("project_id")
    base = base.withColumn("zone_count", (F.lit(MIN_ZONES) + (F.rand(SEED + 20) * (MAX_ZONES - MIN_ZONES))).cast("int"))
    base = base.withColumn("zone_seq", F.explode(F.sequence(F.lit(1), F.col("zone_count"))))

    loc = base.withColumn(
        "location_id", F.concat(F.col("project_id"), F.lit("-L"), F.lpad(F.col("zone_seq").cast("string"), 2, "0"))
    )
    loc = loc.withColumn("zone_name", F.element_at(lit_array(zones), ((F.col("zone_seq") - 1) % len(zones)) + 1))
    loc = loc.withColumn("x_coord", F.round(F.rand(SEED + 21) * 100, 2))
    loc = loc.withColumn("y_coord", F.round(F.rand(SEED + 22) * 100, 2))
    # Stack floors vertically: ~4 meters of height per zone sequence step
    loc = loc.withColumn("z_coord", F.round((F.col("zone_seq") - 1) * 4.0, 2))

    return loc.select("location_id", "project_id", "zone_name", "x_coord", "y_coord", "z_coord")


print("\u2705 create_locations() defined")


### 3.3 `create_tasks()`

Generates **10–25 tasks per project** as a **linear P6-style dependency chain**: every task depends on the previous one (`dependency_task_id`). Delay accumulates with the task sequence and the project's `delay_factor`, so **late-stage tasks slip the most** — exactly how real schedule risk compounds. Each task is mapped to one of its project's zones via `location_id`.

In [ ]:
def create_tasks(spark: SparkSession, projects_df: DataFrame, locations_df: DataFrame) -> DataFrame:
    """Generate P6-style tasks with dependency chains and cascading delays.
    Each task is placed in a zone that matches its TASK_ZONE_AFFINITY (construction
    logic) instead of a round-robin index, so the 3D digital twin reads realistically.
    Carries helper column task_delay_days (consumed by create_costs)."""

    base = projects_df.select("project_id", "start_date", "delay_factor")
    base = base.withColumn("task_count", (F.lit(MIN_TASKS) + (F.rand(SEED + 10) * (MAX_TASKS - MIN_TASKS))).cast("int"))
    base = base.withColumn("task_seq", F.explode(F.sequence(F.lit(1), F.col("task_count"))))

    t = base.withColumn(
        "task_id", F.concat(F.col("project_id"), F.lit("-T"), F.lpad(F.col("task_seq").cast("string"), 3, "0"))
    )
    t = t.withColumn("task_name", F.element_at(lit_array(TASK_NAMES), ((F.col("task_seq") - 1) % len(TASK_NAMES)) + 1))

    # P6-style dependency: each task depends on the previous task (null for the first)
    t = t.withColumn(
        "dependency_task_id",
        F.when(
            F.col("task_seq") > 1,
            F.concat(F.col("project_id"), F.lit("-T"), F.lpad((F.col("task_seq") - 1).cast("string"), 3, "0")),
        ).otherwise(F.lit(None).cast("string")),
    )

    # Planned schedule: sequential tasks, ~20 day duration, staggered by 18 days
    t = t.withColumn("planned_start_date", F.date_add(F.col("start_date"), (F.col("task_seq") - 1) * 18))
    t = t.withColumn("planned_end_date", F.date_add(F.col("planned_start_date"), 20))

    # Cascading delay: grows with task sequence AND project delay_factor
    t = t.withColumn("task_delay_days", (F.col("delay_factor") * F.col("task_seq") * 2).cast("int"))
    t = t.withColumn("actual_end_date", F.date_add(F.col("planned_end_date"), F.col("task_delay_days")))

    t = t.withColumn(
        "task_status",
        F.when(F.col("task_delay_days") <= 0, F.lit("Complete"))
         .when(F.col("task_delay_days") < 15, F.lit("On Track"))
         .when(F.col("task_delay_days") < 40, F.lit("At Risk"))
         .otherwise(F.lit("Delayed")),
    )

    # --- Zone placement by construction affinity (replaces round-robin) --------------
    # Small lookup DataFrames built from the TASK_ZONE_AFFINITY / ZONE_HEIGHT dicts.
    affinity_df = spark.createDataFrame(
        [(tn, zn) for tn, zns in TASK_ZONE_AFFINITY.items() for zn in zns],
        ["task_name", "aff_zone_name"],
    )
    zone_height_df = spark.createDataFrame(
        [(zn, h) for zn, h in ZONE_HEIGHT.items()], ["zh_zone_name", "zone_height"]
    )
    # Target height per task = height of its PRIMARY (first) affinity zone.
    target_df = spark.createDataFrame(
        [(tn, ZONE_HEIGHT[zns[0]]) for tn, zns in TASK_ZONE_AFFINITY.items()],
        ["task_name", "target_height"],
    )

    task_keys = t.select("task_id", "project_id", "task_name")

    # Primary pick: candidate = an affinity zone that ACTUALLY exists in the project.
    # Deterministic + reproducible pick via seeded xxhash64 (spreads tasks across
    # their affinity zones without relying on partition-sensitive F.rand).
    candidates = (
        task_keys.join(affinity_df, "task_name")
        .join(
            locations_df.select("project_id", F.col("zone_name").alias("aff_zone_name"), "location_id"),
            ["project_id", "aff_zone_name"],
        )
    )
    w_primary = Window.partitionBy("task_id").orderBy(
        F.xxhash64(F.concat_ws("|", F.col("task_id"), F.col("aff_zone_name"), F.lit(str(SEED))))
    )
    primary = (
        candidates.withColumn("rn", F.row_number().over(w_primary))
        .filter(F.col("rn") == 1)
        .select("task_id", F.col("location_id").alias("primary_location_id"))
    )

    # Fallback (nearest-by-height) for tasks whose affinity zones are all absent from
    # the project (e.g. Roofing on a project without a Roof zone). Pick the reachable
    # zone whose height is closest to the task's target height; ties broken by zone
    # name (deterministic) -> Roofing -> Floor 3, Landscaping -> Basement.
    proj_zone = locations_df.select("project_id", "zone_name", "location_id").join(
        zone_height_df, F.col("zone_name") == F.col("zh_zone_name")
    )
    fb_base = task_keys.join(primary, "task_id", "left_anti")
    fb = (
        fb_base.join(target_df, "task_name")
        .join(proj_zone, "project_id")
        .withColumn("height_dist", F.abs(F.col("zone_height") - F.col("target_height")))
    )
    w_fb = Window.partitionBy("task_id").orderBy(F.col("height_dist").asc(), F.col("zone_name").asc())
    fallback = (
        fb.withColumn("rn", F.row_number().over(w_fb))
        .filter(F.col("rn") == 1)
        .select("task_id", F.col("location_id").alias("fallback_location_id"))
    )

    t = (
        t.join(primary, "task_id", "left")
        .join(fallback, "task_id", "left")
        .withColumn("location_id", F.coalesce(F.col("primary_location_id"), F.col("fallback_location_id")))
    )
    # --------------------------------------------------------------------------------

    return t.select(
        "task_id", "project_id", "task_name", "planned_start_date", "planned_end_date",
        "actual_end_date", "dependency_task_id", "task_status", "location_id",
        "task_delay_days",  # helper column (dropped before save)
    )


print("\u2705 create_tasks() defined")


### 3.4 `create_costs()`

Generates an **ERP-style** cost breakdown — one row per task × **labor / materials / equipment**. Planned cost varies by **project type** (healthcare and infrastructure cost more), and **actual cost rises with task delay**, so schedule slippage directly drives **cost overruns**.

In [ ]:
def create_costs(spark: SparkSession, tasks_df: DataFrame, projects_df: DataFrame) -> DataFrame:
    """Generate ERP-style costs. Delayed tasks incur higher actual cost; planned
    cost varies by project type."""

    categories = ["labor", "materials", "equipment"]

    c = tasks_df.select("project_id", "task_id", "task_delay_days")
    c = c.withColumn("cost_category", F.explode(lit_array(categories)))
    c = c.join(projects_df.select("project_id", "project_type"), "project_id", "left")

    # Base cost differs by category
    c = c.withColumn(
        "base_cost",
        F.when(F.col("cost_category") == "labor", F.lit(50000))
         .when(F.col("cost_category") == "materials", F.lit(80000))
         .otherwise(F.lit(35000)),
    )

    # Cost pattern varies by project type
    c = c.withColumn(
        "type_mult",
        F.when(F.col("project_type") == "healthcare", F.lit(1.6))
         .when(F.col("project_type") == "infrastructure", F.lit(1.4))
         .when(F.col("project_type") == "commercial", F.lit(1.2))
         .otherwise(F.lit(1.0)),
    )

    # Planned cost with +/-30% natural variation
    c = c.withColumn(
        "planned_cost",
        F.round(F.col("base_cost") * F.col("type_mult") * (F.lit(0.7) + F.rand(SEED + 30) * 0.6), 2),
    )

    # Delay-driven overrun: 1% per delay day plus a small random component
    c = c.withColumn("overrun_pct", F.col("task_delay_days") * 0.01 + F.rand(SEED + 31) * 0.05)
    c = c.withColumn("actual_cost", F.round(F.col("planned_cost") * (F.lit(1.0) + F.col("overrun_pct")), 2))

    # Deterministic, unique cost id: task + category prefix
    c = c.withColumn("cost_id", F.concat(F.col("task_id"), F.lit("-"), F.upper(F.substring(F.col("cost_category"), 1, 3))))

    return c.select("cost_id", "project_id", "task_id", "cost_category", "planned_cost", "actual_cost")


print("\u2705 create_costs() defined")

### 3.4b Enforcing project ↔ task status consistency

`create_projects` derives a project's `status` from its `delay_factor`, while
`create_tasks` derives each `task_status` from that task's own `task_delay_days`.
Because the two are computed **independently**, a project can be labelled
**Delayed** while *none* of its tasks are Delayed — confusing in a demo ("this
project is late, but every task looks fine"). In the pre-cleanup data this
affected **6 of 8** Delayed projects.

Decoupled parent/child status is a data-modeling **anti-pattern**: a roll-up
status should always be reconcilable with the rows beneath it.
`enforce_delayed_consistency` closes the gap — for every Delayed project it bumps
the **latest-sequence** tasks (the ones a cascading schedule delays most) until at
least `min(3, task_count)` are Delayed, keeping `task_delay_days`,
`actual_end_date` and `task_status` in sync. It runs **before** `create_costs` so
the resulting cost overruns reflect the bumped delays.


In [ ]:
def enforce_delayed_consistency(projects_df: DataFrame, tasks_df: DataFrame) -> DataFrame:
    """Guarantee that every Delayed project has at least min(3, task_count) Delayed
    tasks by bumping its latest-sequence tasks. task_delay_days, actual_end_date and
    task_status are updated together so each bumped row stays internally consistent.
    Run BEFORE create_costs so cost overruns reflect the bumped delays."""

    meta = projects_df.select("project_id", F.col("status").alias("project_status"), "delay_factor")
    counts = tasks_df.groupBy("project_id").agg(F.count("*").alias("task_count"))

    # task_id encodes a zero-padded sequence, so ordering by task_id DESC yields the
    # latest-sequence tasks first (the ones a cascading schedule delays most).
    w = Window.partitionBy("project_id").orderBy(F.col("task_id").desc())

    enriched = (
        tasks_df.join(meta, "project_id").join(counts, "project_id")
        .withColumn("rev_rank", F.row_number().over(w))
        .withColumn("bump_n", F.least(F.lit(3), F.col("task_count")))
        .withColumn(
            "is_bumped",
            (F.col("project_status") == "Delayed") & (F.col("rev_rank") <= F.col("bump_n")),
        )
    )

    # Bumped delay reflects how far past the 0.66 Delayed threshold the project sits.
    bumped_delay = F.greatest(
        F.col("task_delay_days"), (F.lit(40) + (F.col("delay_factor") * 20)).cast("int")
    )
    enriched = enriched.withColumn(
        "task_delay_days", F.when(F.col("is_bumped"), bumped_delay).otherwise(F.col("task_delay_days"))
    )
    enriched = enriched.withColumn(
        "actual_end_date",
        F.when(F.col("is_bumped"), F.date_add(F.col("planned_end_date"), F.col("task_delay_days")))
         .otherwise(F.col("actual_end_date")),
    )
    enriched = enriched.withColumn(
        "task_status", F.when(F.col("is_bumped"), F.lit("Delayed")).otherwise(F.col("task_status"))
    )

    # Return exactly the input schema (drops the helper join columns).
    return enriched.select(tasks_df.columns)


print("\u2705 enforce_delayed_consistency() defined")


### 3.5 Generate the raw tables

We call the functions in dependency order: **projects → locations → tasks → costs**. Tasks need locations (for `location_id`) and projects (for schedule), and costs need tasks (for delay) and projects (for type).

In [ ]:
projects_raw = create_projects(spark)
locations_df = create_locations(spark, projects_raw)
tasks_raw = create_tasks(spark, projects_raw, locations_df)
# Reconcile project-level Delayed status with task-level status before costs, so
# cost overruns reflect the bumped delays.
tasks_raw = enforce_delayed_consistency(projects_raw, tasks_raw)
costs_df = create_costs(spark, tasks_raw, projects_raw)

print(f"\u2705 Raw generation complete")
print(f"   projects : {projects_raw.count()}")
print(f"   tasks    : {tasks_raw.count()}")
print(f"   costs    : {costs_df.count()}")
print(f"   locations: {locations_df.count()}")


## 4. Relationship Modeling

The relationships are **encoded in the data itself**, not just documented:

- **projects → tasks**: every task carries its parent `project_id`.
- **tasks → tasks** (dependency chain): `dependency_task_id` points to the immediately preceding task. Because each task's delay grows with its sequence position, a slip early in the chain effectively **cascades** into every downstream task.
- **tasks → costs**: each task fans out into labor / materials / equipment rows; the task's delay raises its `actual_cost`.
- **tasks → locations**: each task maps to one of its project's zones via `location_id`.

Let's verify the **dependency chain and cascading delay** for a single project.

In [ ]:
# Inspect the dependency chain + cascading delay for the first project.
sample_pid = projects_raw.orderBy("project_id").first()["project_id"]

print(f"Dependency chain for project {sample_pid} (note how delay grows down the chain):\n")
(
    tasks_raw.filter(F.col("project_id") == sample_pid)
    .orderBy("task_id")
    .select("task_id", "task_name", "dependency_task_id", "task_delay_days", "task_status", "location_id")
    .show(30, truncate=False)
)

print("\u2705 Relationships verified: dependencies link tasks and delays cascade by sequence")

## 5. Data Realism Enhancements

Perfectly clean data is unrealistic and makes downstream demos look artificial. We deliberately introduce real-world imperfections:

| Imperfection | Where | Why it matters |
| --- | --- | --- |
| **Missing `actual_end_date`** | On-Track projects (still in progress) | Reflects active work with no completion date yet; tests null-handling in BI / AI |
| **Missing `actual_end_date`** | ~40% of On-Track tasks | Mirrors incomplete schedule reporting in the field |
| **Null `dependency_task_id`** | First task of every chain | Genuinely has no predecessor |
| **Outlier projects** | First two projects | Extreme delays + large cost overruns stress-test risk models |

These imperfections are added in a single, isolated function so the generation logic stays clean.

In [ ]:
def add_realism(projects_df: DataFrame, tasks_df: DataFrame):
    """Inject non-critical missing values to mimic real-world reporting gaps.
    Outliers are already baked in via the is_outlier flag in create_projects."""

    # Ongoing (On Track) projects have not finished -> no actual_end_date yet
    projects_enriched = projects_df.withColumn(
        "actual_end_date",
        F.when(F.col("status") == "On Track", F.lit(None).cast("date")).otherwise(F.col("actual_end_date")),
    )

    # ~40% of On Track tasks are still in progress -> null actual_end_date
    tasks_enriched = tasks_df.withColumn(
        "actual_end_date",
        F.when(
            (F.col("task_status") == "On Track") & (F.rand(SEED + 40) < 0.4),
            F.lit(None).cast("date"),
        ).otherwise(F.col("actual_end_date")),
    )

    return projects_enriched, tasks_enriched


projects_df, tasks_df = add_realism(projects_raw, tasks_raw)

missing_proj = projects_df.filter(F.col("actual_end_date").isNull()).count()
missing_task = tasks_df.filter(F.col("actual_end_date").isNull()).count()
outliers = projects_df.filter(F.col("is_outlier")).count()
print(f"\u2705 Realism applied: {missing_proj} projects and {missing_task} tasks have null actual_end_date; {outliers} outlier projects")

## 6. Writing Data to Delta Tables

We persist all four tables to the attached **Fabric Lakehouse** as **Delta** tables using `saveAsTable()` with **overwrite** mode (idempotent — safe to re-run).

Before saving, we **select only the final schema columns**, which automatically drops the internal helper columns (`delay_factor`, `is_outlier`, `task_delay_days`).

In [ ]:
# Final, clean schemas (helper columns excluded by selection)
projects_final = projects_df.select(
    "project_id", "project_name", "project_type", "location", "start_date",
    "planned_end_date", "actual_end_date", "status", "risk_score", "is_outlier",
)
tasks_final = tasks_df.select(
    "task_id", "project_id", "task_name", "planned_start_date", "planned_end_date",
    "actual_end_date", "dependency_task_id", "task_status", "location_id",
)
costs_final = costs_df.select("cost_id", "project_id", "task_id", "cost_category", "planned_cost", "actual_cost")
locations_final = locations_df.select("location_id", "project_id", "zone_name", "x_coord", "y_coord", "z_coord")


def write_delta(df: DataFrame, table_name: str) -> None:
    """Write a DataFrame as a Delta table in overwrite mode (idempotent)."""
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"\u2705 Wrote Delta table: {table_name} ({df.count()} rows)")


write_delta(projects_final, "dbo.projects")
write_delta(tasks_final, "dbo.tasks")
write_delta(costs_final, "dbo.costs")
write_delta(locations_final, "dbo.locations")

print("\u2705 All Delta tables written successfully")

## 7. Validation & Summary

We read the tables **back from the Lakehouse** (not the in-memory DataFrames) to confirm the data persisted correctly, then run a multi-table join and a portfolio insight.

In [ ]:
# 7.1 Row counts read back from the Lakehouse
for tbl in ["dbo.projects", "dbo.tasks", "dbo.costs", "dbo.locations"]:
    print(f"{tbl:<12}: {spark.table(tbl).count():>6} rows")

print("\n\u2705 Row counts validated from persisted Delta tables")

In [ ]:
# 7.2 Join example: projects -> tasks -> costs
projects_t = spark.table("dbo.projects")
tasks_t = spark.table("dbo.tasks")
costs_t = spark.table("dbo.costs")

joined = (
    projects_t.alias("p")
    .join(tasks_t.alias("t"), "project_id")
    .join(costs_t.alias("c"), ["project_id", "task_id"])
    .select(
        "p.project_id", "p.project_name", "p.project_type",
        "t.task_name", "t.task_status",
        "c.cost_category", "c.planned_cost", "c.actual_cost",
    )
)

print("Sample joined rows (projects \u2192 tasks \u2192 costs):\n")
joined.show(10, truncate=False)
print("\u2705 Multi-table join validated")

In [ ]:
# 7.3 Insight: projects with the highest delays AND cost overruns
project_costs = costs_t.groupBy("project_id").agg(
    F.round(F.sum("planned_cost"), 2).alias("total_planned_cost"),
    F.round(F.sum("actual_cost"), 2).alias("total_actual_cost"),
)
project_costs = project_costs.withColumn(
    "cost_overrun", F.round(F.col("total_actual_cost") - F.col("total_planned_cost"), 2)
)

insight = (
    projects_t.join(project_costs, "project_id")
    # Open projects have no actual_end_date yet -> measure delay against today
    .withColumn("delay_days", F.datediff(F.coalesce(F.col("actual_end_date"), F.current_date()), F.col("planned_end_date")))
    .select("project_id", "project_name", "project_type", "status", "risk_score", "delay_days", "cost_overrun")
    .orderBy(F.col("cost_overrun").desc())
)

print("Top projects by cost overrun (note correlation with delay & risk_score):\n")
insight.show(10, truncate=False)
print("\u2705 Portfolio insight generated")

## 8. Refactoring & Code Organization

Everything above is intentionally modular. This final section consolidates the pipeline into **two clean entry points** so the dataset can be (re)built and persisted in one line — ideal for scheduling, testing, or reuse from another notebook.

- `build_all_tables(spark)` → returns a dictionary of the four final DataFrames.
- `save_all_tables(tables)` → persists them as Delta tables.

No logic is duplicated — these orchestrators simply compose the single-purpose functions defined earlier.

In [ ]:
def build_all_tables(spark: SparkSession) -> dict:
    """Build the full synthetic dataset and return final, save-ready DataFrames.
    Composes the modular create_* functions and applies realism in one place."""
    projects_raw = create_projects(spark)
    locations = create_locations(spark, projects_raw)
    tasks_raw = create_tasks(spark, projects_raw, locations)
    # Reconcile project/task Delayed status before costs (see enforce_delayed_consistency).
    tasks_raw = enforce_delayed_consistency(projects_raw, tasks_raw)
    costs = create_costs(spark, tasks_raw, projects_raw)

    projects, tasks = add_realism(projects_raw, tasks_raw)

    return {
        "projects": projects.select(
            "project_id", "project_name", "project_type", "location", "start_date",
            "planned_end_date", "actual_end_date", "status", "risk_score", "is_outlier",
        ),
        "tasks": tasks.select(
            "task_id", "project_id", "task_name", "planned_start_date", "planned_end_date",
            "actual_end_date", "dependency_task_id", "task_status", "location_id",
        ),
        "costs": costs.select("cost_id", "project_id", "task_id", "cost_category", "planned_cost", "actual_cost"),
        "locations": locations.select("location_id", "project_id", "zone_name", "x_coord", "y_coord", "z_coord"),
    }


def save_all_tables(tables: dict) -> None:
    """Persist a dict of {table_name: DataFrame} as Delta tables (overwrite)."""
    for name, df in tables.items():
        write_delta(df, f"dbo.{name}")


print("\u2705 Refactored orchestrators defined: build_all_tables(), save_all_tables()")


In [ ]:
# One-line, reproducible end-to-end rebuild (uncomment to run the consolidated pipeline):
# tables = build_all_tables(spark)
# save_all_tables(tables)

print("\u2705 Notebook complete \u2014 synthetic construction intelligence dataset is ready in the Lakehouse")

## 9. Data Quality Validation (Task 7.1.1)

These assertions guard the three fixes made in this cleanup. Run them **after**
the tables are written to the Lakehouse (Section 6). Each cell fails loudly if an
invariant is violated:

1. **Placement** — every task sits in one of its `TASK_ZONE_AFFINITY` zones, unless
   the project has none of them (then the nearest-by-height fallback is allowed).
2. **Delayed consistency** — every Delayed project has at least `min(3, task_count)`
   Delayed tasks.
3. **Row-count invariants** — 20 projects, 116 locations, 300–350 tasks, and
   costs = 3 × tasks — plus placement-distribution summaries.


In [ ]:
# 9.1 Every task's zone respects TASK_ZONE_AFFINITY, UNLESS the project contains
# none of that task's affinity zones (nearest-by-height fallback is then allowed).
_tasks = spark.table("dbo.tasks")
_locs = spark.table("dbo.locations")

_aff = spark.createDataFrame(
    [(tn, zn) for tn, zns in TASK_ZONE_AFFINITY.items() for zn in zns], ["task_name", "zone_name"]
)

_task_zone = _tasks.select("task_id", "project_id", "task_name", "location_id").join(
    _locs.select("location_id", "zone_name"), "location_id"
)
# Projects that contain at least one of a given task's affinity zones.
_proj_has_aff = (
    _locs.select("project_id", "zone_name").distinct()
    .join(_aff, "zone_name")
    .select("project_id", "task_name").distinct()
    .withColumn("project_has_affinity", F.lit(True))
)
_checked = (
    _task_zone
    .join(_aff.withColumn("in_affinity", F.lit(True)), ["task_name", "zone_name"], "left")
    .join(_proj_has_aff, ["project_id", "task_name"], "left")
)
# Violation = placed outside affinity WHILE the project did have an affinity zone.
_violations = _checked.filter(F.col("in_affinity").isNull() & F.col("project_has_affinity").isNotNull())
_n_viol = _violations.count()
_n_fallback = _checked.filter(F.col("project_has_affinity").isNull()).count()
assert _n_viol == 0, f"{_n_viol} task(s) placed outside affinity without a fallback exemption"
print(f"9.1 PASS - affinity respected. Violations: {_n_viol}; fallback-exempt placements: {_n_fallback}")


In [ ]:
# 9.2 Every Delayed project has >= min(3, task_count) Delayed tasks.
_projects = spark.table("dbo.projects")
_tasks = spark.table("dbo.tasks")

_agg = _tasks.groupBy("project_id").agg(
    F.count("*").alias("task_count"),
    F.sum(F.when(F.col("task_status") == "Delayed", 1).otherwise(0)).alias("delayed_tasks"),
)
_chk = (
    _projects.filter(F.col("status") == "Delayed").select("project_id", "project_name")
    .join(_agg, "project_id")
    .withColumn("required", F.least(F.lit(3), F.col("task_count")))
)
_bad = _chk.filter(F.col("delayed_tasks") < F.col("required"))
_n_bad = _bad.count()
assert _n_bad == 0, f"{_n_bad} Delayed project(s) have fewer than the required Delayed tasks"
print(f"9.2 PASS - all {_chk.count()} Delayed projects satisfy delayed_tasks >= min(3, task_count)")
_chk.orderBy("project_id").show(50, truncate=False)


In [ ]:
# 9.3 Row-count invariants + distribution summaries.
_n_projects = spark.table("dbo.projects").count()
_n_locations = spark.table("dbo.locations").count()
_n_tasks = spark.table("dbo.tasks").count()
_n_costs = spark.table("dbo.costs").count()

assert _n_projects == 20, f"expected 20 projects, got {_n_projects}"
assert _n_locations == 116, f"expected 116 locations, got {_n_locations}"
assert 300 <= _n_tasks <= 350, f"tasks {_n_tasks} outside expected 300-350"
# One cost row per task x {labor, materials, equipment} => costs == 3 x tasks.
assert _n_costs == _n_tasks * 3, f"costs {_n_costs} != tasks*3 ({_n_tasks * 3})"
print(f"9.3 PASS - projects={_n_projects}, locations={_n_locations}, tasks={_n_tasks}, "
      f"costs={_n_costs} (= 3 x tasks)")

print("\nTask placements per zone:")
(
    spark.table("dbo.tasks").alias("t")
    .join(spark.table("dbo.locations").alias("l"), "location_id")
    .groupBy("l.zone_name")
    .agg(F.count("*").alias("task_placements"))
    .orderBy(F.col("task_placements").desc())
    .show(truncate=False)
)

print("Key logical tasks -> zones they now land in (should be single, sensible zones):")
(
    spark.table("dbo.tasks").alias("t")
    .join(spark.table("dbo.locations").alias("l"), "location_id")
    .filter(F.col("t.task_name").isin("Excavation", "Foundation", "Roofing", "Landscaping"))
    .groupBy("t.task_name", "l.zone_name")
    .agg(F.count("*").alias("cnt"))
    .orderBy("t.task_name", F.col("cnt").desc())
    .show(50, truncate=False)
)
